In [17]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from dataset import get_flattened_data, class_names
from pathlib import Path

BASE_DIR = Path.cwd().parent
RESULTS_DIR = BASE_DIR / "results"



In [2]:
X_train, y_train, X_val, y_val, X_test, y_test = get_flattened_data()

# Hyperparameter tuning
c_values = [0.001, 0.01, 0.1, 1.0, 10.0]

validation_accuracies = []
best_c = None
best_accuracy = 0
best_model = None

for c in c_values:
    # lbfgs solver is standard, but saga is often faster for large datasets.
    # max_iter is set to 200 to help with convergence on 784 features.
    model = LogisticRegression(
        C = c,
        solver = "lbfgs",
        max_iter = 1000,
        random_state = 42,
    )

    model.fit(X_train, y_train)

    val_predictions = model.predict(X_val)
    val_accuracy = accuracy_score(y_val, val_predictions)
    validation_accuracies.append(val_accuracy)

    print(f"C = {c:7.3f}, validation accuracy = {val_accuracy:.4f}")

    if val_accuracy > best_accuracy:
        best_accuracy = val_accuracy
        best_c = c
        best_model = model



C =   0.001, validation accuracy = 0.8280
C =   0.010, validation accuracy = 0.8569
C =   0.100, validation accuracy = 0.8627
C =   1.000, validation accuracy = 0.8609
C =  10.000, validation accuracy = 0.8568


C:\Users\William\PycharmProjects\cs178\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [18]:
# Validation accuracy vs C value plot
plt.figure(figsize = (8, 4))
# Using a log scale for the x-axis since C values are logarithmic
plt.plot(c_values, validation_accuracies, marker = 'o')
plt.xscale('log')
plt.xlabel("Regularization parameter (C)")
plt.ylabel("Validation accuracy")
plt.title("Logistic Regression Validation Accuracy by C")
plt.grid(True, which = "both", ls = "--", alpha = 0.5)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "lr_c_vs_accuracy.png")

plt.close()

# Test evaluation using the best model
test_predictions = best_model.predict(X_test)
test_accuracy = accuracy_score(y_test, test_predictions)

# Confusion Matrix
cm = confusion_matrix(y_test, test_predictions)
display = ConfusionMatrixDisplay(
    confusion_matrix = cm,
    display_labels = class_names,
)

# Adjust plot size to prevent overlapping labels
fig, ax = plt.subplots(figsize = (10, 8))
display.plot(ax = ax, xticks_rotation = 45)
plt.title("Logistic Regression Confusion Matrix")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "lr_confusion_matrix.png")

plt.close()

print()
print(f"Best C parameter : {best_c}")
print(f"Best val accuracy: {best_accuracy:.4f}")
print(f"Test accuracy    : {test_accuracy:.4f}")




Best C parameter : 0.1
Best val accuracy: 0.8627
Test accuracy    : 0.8448
